In [2]:
import random

def dynamic_channel_allocation():
    # taking input for total number of channels, show error if index is out of bounds (!(50-100))
    total_channels = int(input("Enter total number of channels (50–100): "))
    if total_channels < 50 or total_channels > 100:
        print("Number of channels must be between 50 and 100.")
        return

    # 10% of total channels is allocated for control channels
    control_channels = total_channels // 10
    voice_channels = total_channels - control_channels
    # display number of channels left
    print(f"\nTotal Channels: {total_channels}")
    print(f"Control Channels Reserved: {control_channels}")
    print(f"Voice/Data Channels Available: {voice_channels}")

    # taking input for cluster size, keep asking until it equals 7
    while True:
        cluster_size = int(input("\nEnter cluster size: "))
        if cluster_size == 7:
            break
        else:
            print("Error: Cluster size must be valid.")

    # creating list to store demands demanded by user per cluster.
    demand_list = []
    print(f"\nEnter channel demand for {cluster_size} cells:")
    for i in range(cluster_size):
        demand = int(input(f"  Enter demand for Cell {i+1}: "))
        demand_list.append(demand)

    # creating copy of demand list for further calculation
    original_demand_list = demand_list[:]

    # summing all demands to keep a tab of total demands for percentage difference
    total_demand_requested = sum(original_demand_list)

    # Create individual channel pool
    channel_pool = list(range(1, voice_channels + 1))

    # Assign random priority channel wise
    channel_priorities = {}
    for ch in channel_pool:
        p = random.randint(1, 5)
        if p in [1, 2]:
            channel_priorities[ch] = ("L", 1)
        elif p == 3:
            channel_priorities[ch] = ("M", 2)
        else:
            channel_priorities[ch] = ("H", 3)

    total_demand = sum(demand_list)
    allocations = [[] for _ in range(cluster_size)]

    if total_demand > voice_channels:
        print(f"\nTotal demand {total_demand} exceeds available {voice_channels}. Adjusting...")

        shares = [0] * cluster_size
        remaining_channels = voice_channels

        for i in range(cluster_size):
            if demand_list[i] < 10:
                shares[i] = demand_list[i]
                remaining_channels -= demand_list[i]

        large_demand_total = sum(d for d in demand_list if d >= 10)
        for i in range(cluster_size):
            if demand_list[i] >= 10:
                share = int((demand_list[i] / large_demand_total) * remaining_channels)
                shares[i] = share

        leftover = remaining_channels - sum(shares[i] for i in range(cluster_size) if demand_list[i] >= 10)
        i = 0
        while leftover > 0:
            if demand_list[i] >= 10:
                shares[i] += 1
                leftover -= 1
            i = (i + 1) % cluster_size

        # Final allocation
        for i in range(cluster_size):
            req = shares[i]
            allocations[i] = channel_pool[:req]
            channel_pool = channel_pool[req:]

    else:
        for i in range(cluster_size):
            req = demand_list[i]
            allocations[i] = channel_pool[:req]
            channel_pool = channel_pool[req:]

    allocated_total = sum(len(a) for a in allocations)
    remaining = voice_channels - allocated_total

    print("\nFinal Voice/Data Channel allocation matrix:")
    for i, channels in enumerate(allocations, start=1):
        labeled = [(ch, channel_priorities[ch][0]) for ch in channels]
        print(f"  Cell {i}: {labeled}")

    print(f"\nTotal allocated: {allocated_total} (of {voice_channels})")
    print(f"Remaining unallocated channels: {remaining}")

    # list for high medium low channels
    high_channels = []
    medium_channels = []
    low_channels = []

    for cell in allocations:
        high_channels.extend([ch for ch in cell if channel_priorities[ch][0] == "H"])
        medium_channels.extend([ch for ch in cell if channel_priorities[ch][0] == "M"])
        low_channels.extend([ch for ch in cell if channel_priorities[ch][0] == "L"])

    print("\nHigh Priority Channels:")
    print(high_channels)

    print("\nMedium Priority Channels:")
    print(medium_channels)

    print("\nLow Priority Channels:")
    print(low_channels)

    # Blocking Probability
    print("\nBlocking Probability per Cell:")
    for i in range(cluster_size):
        demand = original_demand_list[i]
        allocated = len(allocations[i])
        if demand > 0:
            blocking_prob = ((demand - allocated) / demand) * 100
            print(f"  Cell {i+1}: Demand={demand}, Allocated={allocated}, Blocking Probability={blocking_prob:.6f}%")
        else:
            print(f"  Cell {i+1}: Demand=0, Allocated={allocated}, Blocking Probability=N/A")

    # Control Channel Allocation (each shared by 2 cells)
    print("\nControl Channel Allocation:")
    control_pool = list(range(total_channels - control_channels + 1, total_channels + 1))  # e.g., 91–100
    control_allocations = [[] for _ in range(cluster_size)]

    i = 0
    while control_pool:
        ch = control_pool.pop(0)
        control_allocations[i % cluster_size].append(ch)
        control_allocations[(i + 1) % cluster_size].append(ch)
        i += 2

    for i, chans in enumerate(control_allocations, start=1):
        print(f"  Cell {i}: {chans}")


dynamic_channel_allocation()


Enter total number of channels (50–100): 100

Total Channels: 100
Control Channels Reserved: 10
Voice/Data Channels Available: 90

Enter cluster size: 8
Error: Cluster size must be valid.

Enter cluster size: 7

Enter channel demand for 7 cells:
  Enter demand for Cell 1: 10
  Enter demand for Cell 2: 10
  Enter demand for Cell 3: 10
  Enter demand for Cell 4: 23
  Enter demand for Cell 5: 7
  Enter demand for Cell 6: 6
  Enter demand for Cell 7: 5

Final Voice/Data Channel allocation matrix:
  Cell 1: [(1, 'H'), (2, 'M'), (3, 'L'), (4, 'M'), (5, 'H'), (6, 'H'), (7, 'H'), (8, 'H'), (9, 'M'), (10, 'L')]
  Cell 2: [(11, 'H'), (12, 'L'), (13, 'H'), (14, 'L'), (15, 'M'), (16, 'H'), (17, 'L'), (18, 'M'), (19, 'M'), (20, 'L')]
  Cell 3: [(21, 'L'), (22, 'M'), (23, 'M'), (24, 'L'), (25, 'H'), (26, 'H'), (27, 'L'), (28, 'H'), (29, 'M'), (30, 'M')]
  Cell 4: [(31, 'H'), (32, 'L'), (33, 'H'), (34, 'L'), (35, 'M'), (36, 'M'), (37, 'H'), (38, 'H'), (39, 'H'), (40, 'L'), (41, 'L'), (42, 'H'), (43, 